In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.plot_utils import make_histogram

In [ ]:
results_dir = Path("results")
plot_dir = Path("plots")
plot_dir.mkdir(exist_ok=True)
[item.name for item in sorted((results_dir / "eval").iterdir())]

In [ ]:
eval_results = pd.read_csv(results_dir / "eval" / "combined.csv")
eval_results.head()

In [ ]:
groupby_experiment_algo_and_seed = eval_results.groupby(["experiment", "algo", "train_seed"], dropna=False)
seed_reps = groupby_experiment_algo_and_seed.size().rename("n_reps")
timesteps = groupby_experiment_algo_and_seed["timesteps"].mean().rename("mean_total_timesteps")
mean_rewards = groupby_experiment_algo_and_seed["total_reward"].mean().rename("mean_total_reward")
std_rewards = groupby_experiment_algo_and_seed["total_reward"].std().rename("std_total_reward")
pd.concat([seed_reps, timesteps, mean_rewards, std_rewards], axis=1)

In [ ]:
best_results_idx = mean_rewards.groupby(["experiment", "algo"]).idxmax()
best_results = mean_rewards.loc[best_results_idx].rename("best_mean_total_reward")
best_results

In [ ]:
(-best_results.droplevel("train_seed").unstack("algo")).plot.barh(logx=True, title="Average Tracking Errors (100 episodes)")
plt.xlabel("Cost (Negative Total Reward)")
plt.grid()
plt.tight_layout()
plt.savefig(plot_dir / "average_tracking_errors.png", dpi=150)
plt.show()

In [ ]:
# Build lookup: experiment -> best TQC train_seed
best_tqc_seed = {
    exp: idx[2]
    for (exp, algo), idx in best_results_idx.items()
    if algo == 'tqc'
}

# Each row: same environment, without noise then with noise
experiments = [
    ['dL-vL-tqc',  'dL-vL-nL-tqc'],
    ['p2-dL-tqc',  'p2-dL-nL-tqc'],
    ['x2-dL-tqc',  'x2-dL-nL-tqc'],
]

fig, axes = plt.subplots(3, 2, sharey='row', figsize=(7, 6.5))
#fig, axes = plt.subplots(3, 2, sharey=True, figsize=(7, 6.5))

for row_axes, row_experiments in zip(axes, experiments):
    for ax, experiment in zip(row_axes, row_experiments):
        exp_data = eval_results[eval_results['experiment'] == experiment]

        ctrl_rows = exp_data[exp_data['train_seed'].isna()]
        ctrl_algo = ctrl_rows['algo'].iloc[0].upper()
        ctrl_costs = -ctrl_rows['total_reward'].values

        seed = best_tqc_seed[experiment]
        tqc_rows = exp_data[(exp_data['algo'] == 'tqc') & (exp_data['train_seed'] == seed)]
        tqc_costs = -tqc_rows['total_reward'].values

        ax.boxplot([ctrl_costs, tqc_costs], tick_labels=[ctrl_algo, 'TQC'])
        ax.set_title(experiment)
        #ax.set_yscale('log')
        ax.grid(True, axis='y', alpha=0.4)
    row_axes[0].set_ylabel('Cost (-Total Reward)')

fig.suptitle('Tracking Error Distribution (100 episodes, best TQC seed)')
plt.tight_layout()
plt.savefig(plot_dir / 'tracking_errors_boxplots.png', dpi=150)
plt.show()